# MDD snRNA-seq — Disease-Associated Programs in Postmortem Brain

Reproduces **Section 2.6** of the bcNMF paper.

**Setup:** Single-nucleus RNA-seq from postmortem human dorsolateral prefrontal cortex (dlPFC), published by Maitra et al. (2023). The dataset profiles >160,000 nuclei from MDD cases and matched controls. Analysis is restricted to female donors aged 40–60 (Caucasian descent, Douglas Bell–Canada Brain Bank): 4 MDD cases + 4 controls as target; the same 4 controls as background. Focus is on astrocytes (Ast) and oligodendrocytes (Oli).

## Data

- Maitra et al., *Nat. Commun.* **14**, 2912 (2023).
- Available via the Synapse portal (syn accession in the paper) or GEO.
- Place raw count matrices under `data/mdd/` (not tracked by git).

After QC: 26,639 → 18,716 protein-coding genes → top 3,000 HVGs via Scanpy.

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import umap

import sys, os
sys.path.insert(0, os.path.abspath('../..'))
from bcNMF import contrastive_nmf_poisson, nmf_poisson

## 1. Load, filter donors, QC, and select HVGs

In [ ]:
# TODO: load AnnData, filter to female Caucasian donors aged 40-60,
# select 4 cases + 4 controls, restrict to Ast and Oli cell types,
# apply QC, restrict to protein-coding genes, select top 3,000 HVGs
#
# X_target    : np.ndarray (3000, 8886)   -- cases + controls (raw counts)
# X_background: np.ndarray (3000, 3645)   -- controls only (raw counts)
# diagnosis   : array of 'MDD' / 'Control' labels for X_target
# cell_type   : array of 'Ast' / 'Oli' labels for X_target
raise NotImplementedError('Add data loading code here')

## 2. Fit NMF and bcNMF (K = 20, Poisson)

Use Poisson likelihood appropriate for raw counts. Select α via coarse line search on the contrastive reconstruction contrast.

In [ ]:
K = 20
alpha = 1.0  # tune via line search

W_nmf, H_nmf, _ = nmf_poisson(X_target, K=K, niter=300)
W_bc, H_X_bc, H_Y_bc, _ = contrastive_nmf_poisson(X_target, X_background, K=K, alpha=alpha, niter=300)

## 3. UMAP coloured by cell type and diagnosis (Fig. 6a–d)

In [ ]:
reducer = umap.UMAP(random_state=42)
emb_nmf = reducer.fit_transform(H_nmf.T)
emb_bc  = reducer.fit_transform(H_X_bc.T)

# TODO: 2x2 panel — NMF vs bcNMF, coloured by cell_type and diagnosis
pass

## 4. Disease-associated factors and pathway enrichment (Fig. 6e–f)

Rank the 20 bcNMF factors by logistic regression p-value (MDD vs Control). For the top 5, display top-5 gene loadings and run pathway enrichment via Enrichr.

In [ ]:
from scipy import stats
from sklearn.linear_model import LogisticRegression

# TODO: logistic regression of diagnosis on normalised H_X_bc.T,
# rank factors by p-value, extract top-5 genes per top-5 factor
pass